# **Estudo sobre a Localização de Unidades de Atendimento Soroterápico no Estado de São Paulo para Inclusão na Plataforma Onde Tem**

CNPq - PIBITI (processo 183831/2025-0)

Aluna: Julia Graziosi Ortiz, BMACC - ICMC/USP

Orientadora: Maristela Oliveira dos Santos - SME/ICMC/USP

In [ ]:
!pip install adjustText

In [ ]:
import os
from google.colab import userdata

import pandas as pd
import geopandas as gpd

import numpy as np

import mapclassify as mc

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

import seaborn as sns
from adjustText import adjust_text

In [ ]:
os.makedirs("Figuras", exist_ok = True)

##### Importação e Leitura dos Dados

Nome dos (Geo)DataFrames: `municipios`, `postos`, `sinan` e `distancias`

In [ ]:
!gdown --folder 1Q5OtiVFcDMsCDhkNX5oA__ecuBSfloD4 -O "Dados Tratados"

In [ ]:
municipios = gpd.read_file("Dados Tratados/Estruturas_Saude_SP.gpkg")

In [ ]:
# Adiciona as coordenadas como geometria secundária
municipios['coords'] = gpd.points_from_xy(municipios['Longitude'], municipios['Latitude'], crs = 4326)

In [ ]:
municipios.head()

In [ ]:
postos = gpd.read_file("Dados Tratados/PESAs_SP.gpkg")

In [ ]:
postos.head()

In [ ]:
sinan = pd.read_csv("Dados Tratados/SINAN_SP_19A25.csv", index_col = 0)

In [ ]:
sinan.head()

In [ ]:
distancias = pd.read_csv("Dados Tratados/Distancias_API_Municipios_x_Postos.csv", index_col = 0)

In [ ]:
distancias

## **1. Análise dos Acidentes**

In [ ]:
sinan.head()

In [ ]:
tipos_acidente = sinan['Acidente'].unique()

tipos_especie = sinan['Especie'].unique()
tipos_soro = [
    'Botrópico', 'Crotálico', 'Elapídico', 'Laquético',   # Serpentes
    'Fonêutrico', 'Loxoscélico',                          # Aranhas
    'Escorpiônico',                                       # Escorpiões
    'Lonômico'                                            # Lagartas
]

anos = sinan['Ano'].unique()

tipos_gravidade = sinan['Gravidade'].unique()

tipos_tempo = sinan['Tempo'].unique()
tempos_crescente = ['até 1h', '1-3 h', '3-6 h', '6-12 h', '12-24 h', '24 h ou mais']

Contagem de casos

In [ ]:
preliminar = pd.DataFrame(index = tipos_acidente)
preliminar.index.name = 'Acidente'

# Contagem de casos
casos, obitos, uso_soro = [], [], []

for tipo in tipos_acidente:
  df_tipo = sinan[sinan['Acidente'] == tipo]
  casos.append(len(df_tipo))

  filtro_obito = (df_tipo['Evolucao'] ==  'Óbito pelo acidente')
  obitos.append(len(df_tipo[filtro_obito]))

  filtro_soro = (df_tipo['Soroterapia'] == 'Sim')
  uso_soro.append(len(df_tipo[filtro_soro]))

# Insere no df
preliminar[['Casos', 'Óbitos', 'Soroterapia']] = list(zip(casos, obitos, uso_soro))

# Adiciona uma linha com a soma das colunas
preliminar.loc['TOTAL'] = preliminar.sum()

In [ ]:
ano_inicio = sinan['Ano'].min()
ano_fim = sinan['Ano'].max()
print(f' --------- Visão geral dos acidentes no período de {ano_inicio} a {ano_fim} ---------\n')
display(preliminar)

In [ ]:
tab_aci = preliminar.reset_index()

# Remove a linha de total
tab_aci = tab_aci.drop(tab_aci[tab_aci['Acidente'] == 'TOTAL'].index)

# Acidentes em ordem crescente de casos
ordem_cresc_aci = tab_aci[['Acidente', 'Casos']].sort_values('Casos', ascending=True)['Acidente'].tolist()


# Criando os subplots
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]],
    subplot_titles=['Casos', 'Óbitos', 'Soroterapia']
)

# 1. Subplot - Casos
fig.add_trace(
    go.Pie(
        labels=tab_aci['Acidente'],
        values=tab_aci['Casos'],
        textinfo='percent',
        hovertemplate='<b>%{label}</b><br>Casos: %{value:,.0f}<br>Proporção: %{percent}<extra></extra>',
        name='Casos'
    ),
    row=1, col=1
)

# 2. Subplot - Óbitos
fig.add_trace(
    go.Pie(
        labels=tab_aci['Acidente'],
        values=tab_aci['Óbitos'],
        textinfo='percent',
        hovertemplate='<b>%{label}</b><br>Óbitos: %{value:,.0f}<br>Proporção: %{percent}<extra></extra>',
        name='Óbitos'
    ),
    row=1, col=2
)

# 3. Subplot - Uso de Soro
fig.add_trace(
    go.Pie(
        labels=tab_aci['Acidente'],
        values=tab_aci['Soroterapia'],
        textinfo='percent',
        hovertemplate='<b>%{label}</b><br>Aplicação de Soro: %{value:,.0f}<br>Proporção: %{percent}<extra></extra>',
        name='Soros'
    ),
    row=1, col=3
)

# Ajustes de layout
fig.update_layout(
    title=dict(
        text=f'<b>Panorama por Tipo de Acidente</b>',
        x=0.5,
        xanchor='center',
        font=dict(
            size=20,
            family='Arial',
            color='black'
        )
    ),
    height=500,
    width=1100,
    showlegend=True,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.2,
        xanchor='center',
        x=0.5
    )
)

fig.show()

Contagem de casos e óbitos por ano e tipo de acidente

In [ ]:
acidentes_x_ano = pd.DataFrame(index = tipos_acidente)
obitos_x_ano = pd.DataFrame(index = tipos_acidente)

for ano in anos:
  df_ano = sinan[sinan['Ano'] == ano]

  aci_x_ano = df_ano.groupby('Acidente').count()['Ano']
  acidentes_x_ano[ano] = aci_x_ano

  df_obitos = df_ano[df_ano['Evolucao'] == 'Óbito pelo acidente']
  obt_x_ano = df_obitos.groupby('Acidente').count()['Ano']
  obitos_x_ano[ano] = obt_x_ano

In [ ]:
print(' --------- Contagem de acidentes por ano ---------\n')
display(acidentes_x_ano)

print('\n --------- Contagem de óbitos por ano ---------\n')
display(obitos_x_ano)

In [ ]:
# Calculate order of accident types by total cases
ordem_cresc_aci = acidentes_x_ano.sum(axis=1).sort_values(ascending=True)

# Prepare tab_ano for lethality rate calculation
tab_ano = pd.DataFrame(index=anos)
tab_ano['Casos'] = acidentes_x_ano.sum(axis=0)
tab_ano['Obitos'] = obitos_x_ano.sum(axis=0).fillna(0)
tab_ano['Óbitos (%)'] = (tab_ano['Obitos'] / tab_ano['Casos'] * 100).fillna(0)
tab_ano['Grafico obitos'] = tab_ano['Óbitos (%)'].round(2).astype(str) + '%'

# Paleta formal em tons suaves para os tipos de acidente
n_acidentes = len(tipos_acidente)
blues_colors = px.colors.sample_colorscale(
    px.colors.sequential.Blues,
    np.linspace(0.05, 0.85, n_acidentes)
)

# Criar a figura
fig = go.Figure()

for i, tipo_acidente in enumerate(ordem_cresc_aci.index):
    tab_filtrada = acidentes_x_ano.loc[tipo_acidente] # Use acidentes_x_ano

    fig.add_trace(go.Bar(
        x=tab_filtrada.index.to_list(),
        y=tab_filtrada.to_list(),
        name=tipo_acidente,
        marker=dict(
            color=blues_colors[i % len(blues_colors)],
            line=dict(color='white', width=0.5)
        ),
        hovertemplate=(
            '<b>%{x}</b><br>'
            f'{tipo_acidente}: %{{y:,.0f}} casos'
            '<extra></extra>'
        )
    ))

# Linha da taxa de letalidade
fig.add_trace(go.Scatter(
    x=tab_ano["Ano"].to_list() if 'Ano' in tab_ano.columns else tab_ano.index.to_list(),
    y=tab_ano["Óbitos (%)"],
    name="Taxa de letalidade",
    mode="lines+markers+text",
    text=tab_ano["Grafico obitos"],
    textposition="top center",
    texttemplate="<b>%{text}</b>",
    textfont=dict(
        size=14,
        family='Arial',
        color='black',
    ),
    line=dict(
        color='black',
        width=3
    ),
    marker=dict(
        size=8,
        color='black',
        line=dict(color='white', width=0.8)
    ),
    yaxis="y2",
    hovertemplate=(
        '<b>%{x}</b><br>'
        'Taxa de letalidade: %{y:.3f}%'
        '<extra></extra>'
    )
))

# Máximo das barras empilhadas
max_casos_ano = tab_ano['Casos'].max()

# Título
ano_inicio = sinan['Ano'].min()
ano_fim = sinan['Ano'].max()
fig.update_layout(
    title=dict(
        text=f'<b>Evolução anual dos acidentes por animais peçonhentos em São Paulo, {ano_inicio}–{ano_fim}</b>',
        x=0.5,
        xanchor='center',
        font=dict(
            size=20,
            family='Arial',
            color='black'
        )
    ),

    width=1400,
    height=700,
    template='plotly_white',
    paper_bgcolor='white',
    plot_bgcolor='white',
    separators=',.',

    font=dict(
        family='Arial',
        size=14,
        color='black'
    ),

    xaxis=dict(
        title=dict(
            text='<b>Ano</b>',
            font=dict(size=18, family='Arial', color='black')
        ),
        tickmode='array',
        tickvals=list(range(ano_inicio, ano_fim + 1)),
        range=[ano_inicio - 0.5, ano_fim + 0.5],
        tickfont=dict(size=17, family='Arial', color='black'),
        showgrid=False,
        showline=True,
        linewidth=1,
        linecolor='black',
        mirror=True
    ),

    yaxis=dict(
        title=dict(
            text='<b>Número de acidentes</b>',
            font=dict(size=18, family='Arial', color='black')
        ),
        tickfont=dict(size=17, family='Arial', color='black'),
        showgrid=True,
        gridcolor='rgba(0,0,0,0.12)',
        showline=True,
        linewidth=1,
        linecolor='black',
        mirror=True,
        range=[0, max_casos_ano * 1.15]
    ),

    yaxis2=dict(
        title=dict(
            text='<b>Taxa de letalidade (%)</b>',
            font=dict(size=18, family='Arial', color='black')
        ),
        tickfont=dict(size=17, family='Arial', color='black'),
        overlaying='y',
        side='right',
        showgrid=False,
        showline=True,
        linewidth=1,
        linecolor='black',
        range=[0, 1] # Lethality rate is a percentage, so range 0-100. Let's assume it refers to 0-1 (fraction) * 100, if not, adjust.
    ),

    legend=dict(
        title=None,
        orientation='h',
        x=0.5,
        y=-0.18,
        xanchor='center',
        yanchor='top',
        font=dict(size=17, family='Arial', color='black'),
        bgcolor='rgba(255,255,255,0)',
        borderwidth=0
    ),

    barmode='stack',

    margin=dict(t=100, b=150, l=85, r=105)
)

fig.show()


Tempo até o atendimento

Mapa de Vulnerabilidade

In [ ]:
numero_anos = len(sinan['Ano'].unique())

gdf_incidencia_media = municipios[['Codigo Municipio', 'Municipio', 'Codigo Regiao de Saude', 'Regiao de Saude','Populacao Estimada IBGE 2022', 'geometry']].copy()

casos_por_mun = sinan['Origem'].value_counts().reset_index()
casos_por_mun.columns = ['Codigo Municipio', 'Casos']
gdf_incidencia_media = gdf_incidencia_media.merge(casos_por_mun, on = 'Codigo Municipio', how = 'left')

numero_total_casos = gdf_incidencia_media['Casos'].sum()
gdf_incidencia_media['Percentual Casos'] = (gdf_incidencia_media['Casos'] / numero_total_casos) * 100

gdf_incidencia_media['Taxa Media Padronizada'] = ((gdf_incidencia_media['Casos'] / numero_anos) / gdf_incidencia_media['Populacao Estimada IBGE 2022']) * 100000
gdf_incidencia_media.fillna(0, inplace = True)

In [ ]:
def truncar_cmap(cmap_name='YlOrRd', minval=0.15, maxval=0.85, n=256):
    cmap_original = plt.get_cmap(cmap_name)
    novas_cores = cmap_original(np.linspace(minval, maxval, n))
    return mcolors.LinearSegmentedColormap.from_list(f'trunc_{cmap_name}', novas_cores)

cmap_suave = truncar_cmap('YlOrRd', minval=0.15, maxval=0.85)

In [ ]:
classes = mc.NaturalBreaks(gdf_incidencia_media['Taxa Media Padronizada'].dropna())
ax = gdf_incidencia_media.plot(
    column = 'Taxa Media Padronizada',
    legend = True,
    k = classes.k,
    scheme = 'naturalbreaks',
    cmap = cmap_suave,
    figsize = (20, 20),
    legend_kwds = {
        "title": " Taxa média de incidência \n (por 100.000 hab.)",
        #"title": " Average incidence rate \n (per 100,000 inhabitants)",
        "ncols" : 1,
        "loc" : 'lower left',
        "bbox_to_anchor" : (0.05, 0.15),
        'fontsize': 17,          # Tamanho dos rótulos/itens da legenda
        'title_fontsize': 20,    # Tamanho do título da legenda"
    },
    missing_kwds={
        "color": "lightgrey",
        "label": "Nenhum",
    }
)

municipios.boundary.plot(ax = ax, linewidth = 0.5, color = 'black')

ax.axis('off')

titulo = 'Mapeamento da Vulnerabilidade Municipal a Acidentes Peçonhentos'
#titulo = 'Mapping of Municipal Vulnerability to Venomous Accidents'
plt.title(titulo, fontsize = 28, fontweight = "bold")

plt.savefig(f"Figuras/mapa-vulnerabilidade.png", dpi = 300, bbox_inches = 'tight')
plt.show()

## **2. Análise da Rede de Atendimento**

Valores de referência: `COB_ATUAL`, `PERC_ATUAL`

DataFrame: `tab_cobertura_atual`

GeoDataFrame: `cobertura_atual`


### **Descritiva**

In [ ]:
tem_soro = (postos['Tem Soro'] == 1)
postos_com_soro = postos[tem_soro]
cd_mun_com_soro = postos_com_soro['Codigo Municipio'].unique()
print(f'Número de municípios que possuem algum tipo de soro soro: {len(cd_mun_com_soro)}\n')
print(f'Número de PESA: {len(postos_com_soro)}')

In [ ]:
tipos_soro = ['Botrópico', 'Crotálico', 'Elapídico', 'Laquético', 'Loxoscélico', 'Fonêutrico', 'Escorpiônico', 'Lonômico']
cols_tabela = ['PESA', 'Municipio', 'RRAS', 'RS', 'DRS', 'GVE']
tabela_rede_atual = pd.DataFrame(index = tipos_soro, columns = cols_tabela)

for soro in tipos_soro:
  postos_com_soro = postos[postos[soro] == 1]
  n_postos = len(postos_com_soro)

  idx_mun_soro = postos_com_soro['Indice Municipio'].to_list()
  municipios_com_soro = municipios[municipios.index.isin(idx_mun_soro)]

  n_mun  = len(municipios_com_soro)
  n_rras = len(municipios_com_soro['Macrorregiao de Saude'].unique())
  n_rs   = len(municipios_com_soro['Regiao de Saude'].unique())
  n_drs  = len(municipios_com_soro['Departamento Regional de Saude'].unique())
  n_gve  = len(municipios_com_soro['Grupo de Vigilancia Epidemiologica'].unique())

  linha_soro = [n_postos, n_mun, n_rras, n_rs, n_drs, n_gve]
  tabela_rede_atual.loc[soro] = linha_soro

# Valores de referência
total_pesas = len(postos[postos['Tem Soro'] == 1])
total_mun   = len(municipios)
total_rras  = len(municipios['Macrorregiao de Saude'].unique())
total_rs    = len(municipios['Regiao de Saude'].unique())
total_drs   = len(municipios['Departamento Regional de Saude'].unique())
total_gve   = len(municipios['Grupo de Vigilancia Epidemiologica'].unique())

linha_referencia = [total_pesas, total_mun, total_rras, total_rs, total_drs, total_gve]
tabela_rede_atual.loc['REFERÊNCIA'] = linha_referencia

tabela_rede_atual#.to_latex()

In [ ]:
sem_soro_escorpionico = (postos['Tem Soro'] == 1) & (postos['Escorpiônico'].isna())
postos[sem_soro_escorpionico]

In [ ]:
# 1. Configuração de estilo
fig, ax = plt.subplots(figsize=(20, 20))

# Postos ativos e municípios que possuem um posto
postos_com_soro = postos[postos['Tem Soro'] == 1]
idx_mun_com_soro = list(postos_com_soro['Indice Municipio'].unique())
municipios_com_pesa = municipios[municipios.index.isin(idx_mun_com_soro)]

# Plots
municipios_com_pesa.plot(
    ax = ax,
    color = '#1f4e79',
    zorder = 1
)

municipios.boundary.plot(
    ax = ax,
    color = 'dimgray',
    linewidth = 0.5,
    zorder = 2
)

postos_com_soro.plot(
    ax = ax,
    marker = 'o',
    color = '#ffd700',
    markersize = 40,
    edgecolor = 'black',
    linewidth = 1,
    zorder = 3
)

# --- Legendas e Títulos (Mantidos) ---
legenda = [
    Line2D(
        [0], [0],
        marker = 's',
        color = 'w',
        label = 'Município sem PESA',
        markerfacecolor = 'white',
        markersize = 17,
        markeredgecolor = 'black'
    ),
    Line2D(
        [0], [0],
        marker = 's',
        color = 'w',
        label = 'Município com PESA',
        markerfacecolor = '#1f4e79',
        markersize = 17
    ),
    Line2D(
        [0], [0],
        marker = 'o',
        color = 'w',
        label = 'PESA',
        markerfacecolor = '#ffd700',
        markersize = 15,
        markeredgecolor = 'black'
    ),
]

ax.legend(
    handles = legenda,
    loc = 'upper left',
    bbox_to_anchor = (0.05, 0.35),
    fontsize = 25,
    frameon = True
)

ax.axis('off')

plt.title(
    f'Pontos Estratégicos para Soro Antiveneno do Estado de São Paulo',
    fontsize = 30,
    fontweight = "bold"
)

plt.tight_layout()

# plt.savefig("Figuras/rede-atual.png", dpi = 300, bbox_inches = 'tight')
plt.savefig("Figuras/rede-atual.png", dpi = 300, bbox_inches = 'tight')
plt.show()

### **Cobertura**

In [ ]:
cobertura_atual = municipios[['Codigo Municipio', 'Municipio']].copy()

cols_tabela = ['PESAS', 'Tempo Máximo (s)', 'Municípios Cobertos', 'Cobertura (%)']
tab_cobertura_atual = pd.DataFrame(index = tipos_soro, columns = cols_tabela)

tipos_soro = ['Botrópico', 'Crotálico', 'Elapídico', 'Laquético', 'Loxoscélico', 'Fonêutrico', 'Escorpiônico', 'Lonômico']
dict_t_max = {
    'Botrópico': 3000,   # 50 minutos
    'Crotálico': 3000,
    'Elapídico': 3000,
    'Laquético': 3000,
    'Loxoscélico': 3000,
    'Fonêutrico': 3000,
    'Escorpiônico': 3000,
    'Lonômico': 3000
}

# Filtrando postos operantes e ajustando as distâncias
postos_ativos = postos[postos['Tem Soro'] == 1].reset_index().rename(columns = {'index': 'Indice Original'})
indices_originais = postos_ativos['Indice Original'].to_list()
distancias_ativos = distancias.iloc[:, indices_originais]

COB_ATUAL = 0
for soro in tipos_soro:
  cobertura_atual[soro] = 0
  t_max_soro = dict_t_max[soro]
  for m in municipios.index:
    postos_com_soro = postos_ativos[postos_ativos[soro] == 1]
    for p in postos_com_soro.index:
      d = distancias_ativos.iloc[m, p]
      if (d <= t_max_soro) and (d >= 0):
        cobertura_atual.loc[m, soro] = 1
        break

  n_pesas = len(postos_com_soro)
  n_muns = cobertura_atual[soro].sum()
  perc = n_muns / len(municipios) * 100

  linha_soro = [n_pesas, t_max_soro, n_muns, perc]
  tab_cobertura_atual.loc[soro] = linha_soro

  COB_ATUAL += n_muns

# Calcula o percentual de cobertura da rede atual
COB_IDEAL = len(municipios) * len(tipos_soro)
PERC_ATUAL = COB_ATUAL / COB_IDEAL * 100

In [ ]:
print(' --- Cobertura da Rede Atual --- ')
print(f'Total de municípios cobertos: {COB_ATUAL} de {COB_IDEAL}')
print(f'Percentual de cobertura: {COB_ATUAL / COB_IDEAL * 100:.2f}%\n')

display(tab_cobertura_atual)

# **Modelos Matemáticos**

##### Instalação do Gurobi e configuração da licença acadêmica

In [ ]:
!pip install gurobipy

In [ ]:
from gurobipy import Model, GRB, quicksum, Env
import numpy as np

In [ ]:
# Parametros da licença de estudante
parametros = {
  "WLSACCESSID" : userdata.get('GUROBI_WLSACCESSID'),
  "WLSSECRET" : userdata.get('GUROBI_WLSSECRET'),
  "LICENSEID" : int(userdata.get('GUROBI_LICENSEID'))
}

## **1. Modelo de Cobertura**

$$\begin{align}
\min \ \ \ &\sum_{j \ \in \ J} y_j
\tag{1}
\\
\text{s.a.} \ \
&\sum_{j \ \in J} c_{ij} y_{j} \geq 1 &\forall i \in I
\tag{2}
\\
& y_j \in \{0,1\} &\forall j \in J
\tag{3}
\end{align}$$

A função objetivo $(1)$ minimiza o número de postos instalados. As restrições $(2)$ garantem que todos os municípios devem ser atendidos por pelo menos um posto. As restrições $(3)$ representam o domínio das variáveis.

In [ ]:
os.makedirs("Resultados Cobertura", exist_ok = True)

In [ ]:
I = np.arange(len(municipios))
J = np.arange(len(postos))

# Auxiliar: possibilidade de cobertura
t_max = 3000
d = distancias.to_numpy()
c = {}
for i in I:
  for j in J:
      c[i, j] = 1 if d[i,j] <= t_max else 0

In [ ]:
def resolve_modelo_cobertura(gdf_municipios, gdf_postos, df_distancias, t_max, col_restricao = 'Sem restricao'):
  '''
    O parâmetro restricao tem o objetivo de restringir os postos que podem
    ser considerados para cobrir os municípios
  '''
  # Validação da string
  opts_restricao = [
      'Sem restricao',
      'Macrorregiao de Saude',
      'Regiao de Saude',
      'Departamento Regional de Saude',
      'Grupo de Vigilancia Epidemiologica'
  ]
  if col_restricao not in opts_restricao:
    print("Parâmetro 'restricao' inválido.\nEscolha entre:")
    for opt in opts_restricao:
      print(f"- {opt}")
    return None

  aplica_restricao = (col_restricao != 'Sem restricao')

  # -------- Pré-processamento

  # Conjuntos
  I = np.arange(len(gdf_municipios))
  J = np.arange(len(gdf_postos))

  # Parâmetros
  d = df_distancias.to_numpy()
  c = {}
  for i in I:
    # Determina a divisão de saúde escolhida
    div_saude_munic = gdf_municipios.loc[i, col_restricao] if aplica_restricao else False
    for j in J:
      if aplica_restricao:
        idx_mun_posto = gdf_postos.loc[j, 'Indice Municipio']
        div_saude_posto = gdf_municipios.loc[idx_mun_posto, col_restricao]
      else:
        div_saude_posto = div_saude_munic

      respeita_restricao = (div_saude_munic == div_saude_posto)

      c[i, j] = 1 if (d[i,j] <= t_max and respeita_restricao) else 0

  # -------- Modelo
  env = Env(params = parametros)
  modelo = Model(f"Cobertura_Simples_{col_restricao}", env = env)

  # Variável
  y = modelo.addVars(J, vtype=GRB.BINARY, name='posto_instalado')

  # Função Objetivo
  modelo.setObjective(
      quicksum(y[j] for j in J),
      GRB.MINIMIZE
  )

  # Restrições
  modelo.addConstrs(
      (
          quicksum(c[i, j] * y[j] for j in J) >= 1
          for i in I
      ),
      name='R_Cobertura'
  )

  modelo.optimize()

  if modelo.status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    res_y = modelo.getAttr('X', y)
    fo_modelo = modelo.ObjVal


    # Salva os resultados
    df_result = pd.DataFrame({'Nome Posto': gdf_postos['Nome Posto'].values}, index=J)
    for j in J:
      df_result.loc[j, 'Posto Instalado'] = 1 if res_y[j] > 0.5 else 0

    resultado = {
      "status": modelo.status,
      "t_max": t_max,
      "fo_modelo": fo_modelo,
      "df_postos": df_result
    }

  else:
    print('Não encontrou solução para o problema...')
    if modelo.Status == GRB.INFEASIBLE:
      print("Modelo é infactível. Computando o IIS...")

      # Redireciona o Gurobi para focar em encontrar o conflito
      modelo.computeIIS()

      # Salva o resultado no formato .ilp
      modelo.write("modelo_conflito.ilp")
      print("Arquivo 'modelo_conflito.ilp' salvo com sucesso!")
    resultado = None

  modelo.dispose()
  env.dispose()

  return resultado

In [ ]:
resultado_modelo = resolve_modelo_cobertura(
    municipios,
    postos,
    distancias,
    3000,
    'Sem restricao'
)

In [ ]:
def plot_selecionados_cobertura(gdf_municipios, gdf_postos, resultado, nome):

  gdf_m = gdf_municipios[['Codigo Municipio', 'Municipio', 'geometry']].copy()
  gdf_p = gdf_postos[['Nome Posto', 'Tem Soro', 'geometry']].copy()

  df_selecionados = resultado['df_postos']
  gdf_p = gdf_p.merge(df_selecionados, on = 'Nome Posto', how = 'left')
  gdf_p['Nova Instalacao'] = ((gdf_p['Posto Instalado'] == 1) & (gdf_p['Tem Soro'] == 0)).astype(int)
  gdf_p['Posto Removido'] = ((gdf_p['Posto Instalado'] == 0) & (gdf_p['Tem Soro'] == 1)).astype(int)
  gdf_p['Posto Mantido'] = ((gdf_p['Posto Instalado'] == 1) & (gdf_p['Tem Soro'] == 1)).astype(int)

  # Plot
  fig, ax = plt.subplots(figsize=(10, 10))

  gdf_m.boundary.plot(
      ax = ax,
      color = 'dimgray',
      linewidth = 0.5,
      zorder = 0
  )

  postos_novos = gdf_p[gdf_p['Nova Instalacao'] == 1]
  postos_mantidos = gdf_p[gdf_p['Posto Mantido'] == 1]

  postos_novos.plot(
      ax = ax,
      color = '#ff00ff',
      marker = 'o',
      markersize = 40,
      edgecolor = 'black',
      linewidth = 1,
      zorder = 1
  )

  postos_mantidos.plot(
      ax = ax,
      color = '#ffd700',
      marker = 'o',
      markersize = 40,
      edgecolor = 'black',
      linewidth = 1,
      zorder = 2
  )

  # 3. Legenda manual
  legenda_manual = []
  if not postos_mantidos.empty:
    legenda_manual.append(
      Line2D([0], [0], marker='o', color='w', label='PESA Mantido',
      markerfacecolor='#ffd700', markersize=10, markeredgecolor='black'),
    )
  if not postos_novos.empty:
    legenda_manual.append(
        Line2D([0], [0], marker='o', color='w', label='Novo PESA',
      markerfacecolor='#ff00ff', markersize=10, markeredgecolor='black')
    )

  ax.legend(
      handles = legenda_manual,
      loc = 'lower left',
      fontsize = 11,
      bbox_to_anchor = (0.05, 0.05),
      frameon = True
  )

  ax.axis('off')

  plt.tight_layout()

  tempo = resultado['t_max'] / 60
  n_pesas = resultado['fo_modelo']
  n_novas = len(postos_novos)
  titulo = f'Modelo de Cobertura com Distância de {int(tempo)} minutos\n'
  titulo += f'{int(n_pesas)} PESAs selecionados | {n_novas} novas instalações'
  plt.title(titulo, fontsize = 15, fontweight = 'bold')
  plt.savefig(f"modelo-{nome}.png", dpi=300, bbox_inches='tight')

  return

In [ ]:
plot_selecionados_cobertura(municipios, postos, resultado_modelo, 'cobertura-simples')

In [ ]:
def plot_area_cobertura_selecionados(gdf_municipios, gdf_postos, df_distancias, resultado, nome):

  gdf_m = gdf_municipios[['Codigo Municipio', 'Municipio', 'geometry']].copy()
  gdf_p = gdf_postos[['Nome Posto', 'Tem Soro', 'geometry']].copy()

  df_selecionados = resultado['df_postos']
  gdf_p = gdf_p.merge(df_selecionados, on = 'Nome Posto', how = 'left')

  gdf_p['Nova Instalacao'] = ((gdf_p['Posto Instalado'] == 1) & (gdf_p['Tem Soro'] == 0)).astype(int)
  gdf_p['Posto Removido'] = ((gdf_p['Posto Instalado'] == 0) & (gdf_p['Tem Soro'] == 1)).astype(int)

  gdf_selecionados = gdf_p[gdf_p['Posto Instalado'] == 1].reset_index().rename(columns = {'index': 'Indice Original'})
  idx_orig_selecionados = gdf_selecionados['Indice Original'].to_list()
  df_dist_selecionados = df_distancias.iloc[:, idx_orig_selecionados]

  gdf_m['Posto mais Próximo'] = np.nan
  gdf_m['Distancia (s)'] = np.nan

  for m in gdf_m.index:
    # Encontra o nome do posto (coluna) com a distância mínima para o município 'm'
    nome_posto_min = df_dist_selecionados.iloc[m, :].idxmin()

    # Atribui a distância mínima para o município 'm'
    gdf_m.loc[m, 'Distancia (s)'] = df_dist_selecionados.loc[m, nome_posto_min]

    # Encontra o 'Indice Original' do posto com a distância mínima
    idx_orig_min = gdf_selecionados[gdf_selecionados['Nome Posto'] == nome_posto_min]['Indice Original'].iloc[0]
    gdf_m.loc[m, 'Posto mais Próximo'] = idx_orig_min

  area_cobertura = gdf_m.dissolve(by = 'Posto mais Próximo', as_index = False)[['Posto mais Próximo', 'geometry']]

  # Plot
  fig, ax = plt.subplots(figsize=(10, 10))

  area_cobertura.plot(
      ax = ax,
      categorical = True,
      column = 'Posto mais Próximo',
      # cmap = 'Pastel2',
      zorder = 0
  )

  gdf_selecionados.plot(ax = ax, color = 'red', markersize = 75, zorder = 2)
  gdf_selecionados.plot(ax = ax, color = 'white', marker = 'P', zorder = 3)

  ax.axis('off')

  plt.tight_layout()
  plt.savefig(f"modelo-{nome}.png", dpi=300, bbox_inches='tight')

  return gdf_m, gdf_p

In [ ]:
plot_area_cobertura_selecionados(municipios, postos, distancias, resultado_modelo, 'cobertura-simples' )

In [ ]:
resultado_modelo_regiao = resolve_modelo_cobertura(
    municipios,
    postos,
    distancias,
    5400,
    'Regiao de Saude'
)

In [ ]:
plot_selecionados_cobertura(municipios, postos, resultado_modelo_regiao, 'cobertura-rras')

## **2. Modelo de p-Medianas**

$$\begin{align}
\min \ \ \ &\sum_{m \in M} \sum_{p \in P}d_{mp}x_{mp} \\
\text{s.a.} \ \
&\sum_{p \in P} x_{mp} = 1 &\forall m \in M
\\
&x_{mp} \leq y_{p} &\forall m \in M, \forall p \in P
\\
&\sum_{p \in P} y_p = \mathfrak{p} &\forall p \in P
\\
&x_{mp}, y_{p} \in \{0,1\} &\forall m \in M, \forall p \in P
\end{align}$$

## **3. Modelo de Máxima Cobertura Ponderada com Penalização**

In [ ]:
os.makedirs("Resultados Penalização", exist_ok = True)

##### Cálculo das demandas

In [ ]:
# Cálculo das demandas
demandas = municipios[['Municipio', 'Codigo Municipio']].copy()
anos = sinan['Ano'].unique()
soros = ['Botrópico', 'Crotálico', 'Elapídico', 'Laquético', 'Fonêutrico', 'Loxoscélico', 'Escorpiônico',  'Lonômico']

for soro in soros:
  demandas[soro] = np.nan
  df_soro = sinan[(sinan['Especie'] == soro) & (sinan['Soroterapia'] == 'Sim')]
  contagem_casos = df_soro.value_counts('Origem').rename_axis('Codigo Municipio')
  demandas[soro] = demandas['Codigo Municipio'].map(contagem_casos)

  # Demandas para modelo do SBPO: média de casos que usaram soroterapia
  df_soro = sinan[(sinan['Especie'] == soro) & (sinan['Soroterapia'] == 'Sim')]
  contagem_casos = df_soro.value_counts('Origem').rename_axis('Codigo Municipio')
  demandas[soro] = demandas['Codigo Municipio'].map(contagem_casos)
  demandas[soro] = demandas[soro] / len(anos)                                   # média anual
  demandas[soro] = np.ceil(demandas[soro])                                      # arredonda para cima
  demandas[soro] = demandas[soro].fillna(0.5)

  # # Demandas para modelo do CNMAC: média de casos (independente de soroterapia) normalizada pelo máximo
  # df_soro = sinan[sinan['Especie'] == soro]
  # contagem_casos = df_soro.value_counts('Origem').rename_axis('Codigo Municipio')
  # demandas[soro] = demandas['Codigo Municipio'].map(contagem_casos)
  # demandas[soro] = np.ceil(demandas[soro])            # arredonda para cima
  # maior_valor = demandas[soro].max()
  # demandas[soro] = demandas[soro] / maior_valor       # normaliza pelo máximo

# demandas

#### Conjuntos e Parâmetros do modelo

In [ ]:
soros = ['Botrópico', 'Crotálico', 'Elapídico', 'Laquético', 'Fonêutrico', 'Loxoscélico', 'Escorpiônico',  'Lonômico']
postos_ativos = postos[postos['Tem Soro'] == 1].reset_index().rename(columns = {'index': 'Indice Original'})

# Conjuntos
I = np.arange(len(municipios))
J = np.arange(len(postos_ativos))
K = np.arange(len(soros))

In [ ]:
# Configuração atual
h = {(j, k): int(postos_ativos.loc[j, soros[k]] == 1) for j in J for k in K}
valor_p_k = {k: int(postos_ativos[soros[k]].sum()) for k in K}

# Distâncias
indices_originais = postos_ativos['Indice Original'].to_list()
distancias_ativos = distancias.iloc[:, indices_originais]
d = distancias_ativos.to_numpy()

# Auxiliar: possibilidade de cobertura
t_max = 3000
c = {}
for i in I:
  for j in J:
      c[i, j] = 1 if d[i,j] <= t_max else 0

# Demandas
w = np.zeros((len(I), len(K)))
for k, soro in enumerate(soros):
  w[:, k] = demandas[soro].values

#### Modelo no Gurobi

In [ ]:
def roda_modelo_penalizacao(gdf_munic, gdf_postos, lambda_pen):

  env = Env(params = parametros)
  modelo = Model(f"Max_Cob_Pond_LambdaPen_{lambda_pen}", env = env)

  # Variáveis
  x = modelo.addVars(I, K, vtype=GRB.BINARY, name='cobertura_munic_soro')
  y = modelo.addVars(J, K, vtype=GRB.BINARY, name='alocacao_soro_posto')

  # Função Objetivo
  modelo.setObjective(
    quicksum(w[i, k] * x[i, k] for i in I for k in K)
    - lambda_pen * quicksum((1 - h[j, k]) * y[j, k] for j in J for k in K),
    GRB.MAXIMIZE
  )

  # Restrições
  # 1. Cobertura
  modelo.addConstrs(
      (
          x[i, k] <= quicksum(c[i, j] * y[j, k] for j in J)
          for i in I for k in K
      ),
      name='R_Cobertura'
  )

  # 2. Quantidade fixa por soro
  modelo.addConstrs(
      (
          quicksum(y[j, k] for j in J) == valor_p_k[k]
          for k in K
      ),
      name='R_TotalPostosPorSoro'
  )

  # Otimiza o modelo
  modelo.optimize()

  # Salva os resultados
  if modelo.status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    res_x = modelo.getAttr('X', x)
    res_y = modelo.getAttr('X', y)
    fo_modelo = modelo.ObjVal

    # Cobertura "crua": quantidade de municípios cobertos
    fo_cobertura_real = sum(1 for i in I for k in K if res_x[i, k] > 0.5)

    # Cobertura ponderada pela demanda
    fo_cobertura_pond = sum(w[i, k] for i  in I for k in K if res_x[i, k] > 0.5)

    # Número de municípios que possuem cobertura para pelo menos um tipo de soro
    n_munic_total = sum(1 for i in I if any(res_x[i, k] > 0.5 for k in K))

    # Comparação de alocação
    n_novas = sum(1 for j in J for k in K if res_y[j, k] > 0.5 and h[j, k] == 0)
    n_mantidas = sum(1 for j in J for k in K if res_y[j, k] > 0.5 and h[j, k] == 1)

    # DataFrame com resumo por soro
    lista_df = []
    for k, soro in enumerate(soros):
      n_postos = sum(1 for j in J if res_y[j, k] > 0.5)
      n_cobertos = sum(1 for i in I if res_x[i, k] > 0.5)
      n_nao_cobertos = len(I) - n_cobertos
      perc = n_cobertos / len(I) * 100

      lista_df.append({
        'Lambda': lambda_pen,
        'Soro': soro,
        'Postos com soro': n_postos,
        'Tempo máximo (s)': t_max,
        'Municípios cobertos': n_cobertos,
        'Municípios não cobertos': n_nao_cobertos,
        'Cobertura (%)': perc
      })

    df_resultado = pd.DataFrame(lista_df)

    df_result_alocacao = pd.DataFrame({'Nome Posto': gdf_postos['Nome Posto'].values}, index=J)
    df_result_cobertura = pd.DataFrame({'Municipio': gdf_munic['Municipio'].values}, index=I)

    for k, soro in enumerate(soros):
      df_result_alocacao[soro] = np.nan
      for j in J:
        df_result_alocacao.loc[j, soro] = 1 if res_y[j, k] > 0.5 else 0

      df_result_cobertura[soro] = np.nan
      for i in I:
        df_result_cobertura.loc[i, soro] = 1 if res_x[i, k] > 0.5 else 0

    resultado = {
      "lambda_pen": lambda_pen,
      "status": modelo.status,
      "fo_modelo": fo_modelo,
      "fo_cobertura_real": fo_cobertura_real,
      "fo_cobertura_ponderada": fo_cobertura_pond,
      "novas_implantacoes": n_novas,
      "alocacoes_mantidas": n_mantidas,
      "municipios_com_algum_soro": n_munic_total,
      "df_resultado": df_resultado,
      "df_result_alocacao": df_result_alocacao,
      "df_result_cobertura": df_result_cobertura
    }

  else:
    print(f"Modelo inviável para lambda = {lambda_pen}")
    resultado = None


  modelo.dispose()
  env.dispose()

  return resultado

In [ ]:
def salva_resultados_penalizacao(df_objetivo, df_resumo, gdf_cobertura, gdf_alocacao, lambda_pen, resultado, nome):

  # Cria uma linha para ser inserida no df_objetivo
  nova_linha = [lambda_pen, resultado['fo_modelo'], resultado['fo_cobertura_real'],
                resultado['fo_cobertura_ponderada'], resultado['novas_implantacoes'],
                resultado['alocacoes_mantidas'], resultado['municipios_com_algum_soro']
  ]
  df_objetivo.loc[len(df_objetivo)] = nova_linha

  # Adiciona colunas dos resultados
  df_resumo = pd.concat([df_resumo, resultado['df_resultado']], ignore_index = True)

  # Adiciona colunas dos gdfs
  def add_sufixo(df, lambda_pen):
    for soro in soros:
      if soro in df.columns:
        df = df.rename(columns = {soro: f'{soro}_{lambda_pen}'})
    return df
  r_cobertura = resultado['df_result_cobertura']
  r_cobertura = add_sufixo(r_cobertura, lambda_pen)

  r_alocacao = resultado['df_result_alocacao']
  r_alocacao = add_sufixo(r_alocacao, lambda_pen)

  gdf_cobertura = gdf_cobertura.merge(r_cobertura, on = 'Municipio', suffixes = ('', f'_{lambda_pen}'))
  gdf_alocacao = gdf_alocacao.merge(r_alocacao, on = 'Nome Posto', suffixes = ('', f'_{lambda_pen}'))

  # Salva resultados
  df_objetivo.to_csv(f"Resultados Penalização/Objetivo_{nome}.csv", index = False)
  df_resumo.to_csv(f"Resultados Penalização/Resumo_{nome}.csv", index = False)
  gdf_cobertura.to_file(f"Resultados Penalização/Coberturas_{nome}.gpkg", index = False)
  gdf_alocacao.to_file(f"Resultados Penalização/Alocacoes_{nome}.gpkg", index = False)

  return df_objetivo, df_resumo, gdf_cobertura, gdf_alocacao

In [ ]:
# pasta = 'Resultados Penalização'

# for nome_arquivo in os.listdir(pasta):
#   caminho_arquivo = os.path.join(pasta, nome_arquivo)
#   if os.path.isfile(caminho_arquivo):
#     os.remove(caminho_arquivo)

#### Testes para o parâmetro de Penalização

In [ ]:
lambdas_teste_pequenos = np.linspace(0, 2, 11).tolist()

# Arredonda para uma casa decimal
for i in range(len(lambdas_teste_pequenos)):
  lambdas_teste_pequenos[i] = round(lambdas_teste_pequenos[i], 1)

In [ ]:
resumo_obj_peq = pd.DataFrame(columns = ['Lambda', 'FO Cobertura', 'FO Cobertura Real', 'FO Cobertura Ponderada', 'Novas Implantações', 'Alocações Mantidas', 'Municípios com Algum Soro'])
resumo_pequenos = pd.DataFrame(columns = ['Lambda', 'Soro', 'Postos com soro', 'Tempo máximo (s)', 'Municípios cobertos', 'Municípios não cobertos', 'Cobertura (%)'])
coberturas_pequenos = municipios[['Municipio', 'Codigo Municipio', 'geometry']].copy()
alocacoes_pequenos = postos_ativos[['Nome Posto', 'geometry']].copy()

for lambda_pen in lambdas_teste_pequenos:
  results = roda_modelo_penalizacao(municipios, postos_ativos, lambda_pen)
  if results is not None:
    resumo_obj_peq, resumo_pequenos, coberturas_pequenos, alocacoes_pequenos = salva_resultados_penalizacao(resumo_obj_peq, resumo_pequenos, coberturas_pequenos, alocacoes_pequenos, lambda_pen, results, 'Pequenos')

In [ ]:
lambdas_teste_grandes = [2, 3, 5, 8, 10, 15, 20, 100]

In [ ]:
resumo_obj_grandes = pd.DataFrame(columns = ['Lambda', 'FO Cobertura', 'FO Cobertura Real', 'FO Cobertura Ponderada', 'Novas Implantações', 'Alocações Mantidas', 'Municípios com Algum Soro'])
resumo_grandes = pd.DataFrame(columns = ['Lambda', 'Soro', 'Postos com soro', 'Tempo máximo (s)', 'Municípios cobertos', 'Municípios não cobertos', 'Cobertura (%)'])
coberturas_grandes = municipios[['Municipio', 'Codigo Municipio', 'geometry']].copy()
alocacoes_grandes = postos_ativos[['Nome Posto', 'geometry']].copy()

for lambda_pen in lambdas_teste_grandes:
  results = roda_modelo_penalizacao(municipios, postos_ativos, lambda_pen)
  if results is not None:
    resumo_obj_grandes, resumo_grandes, coberturas_grandes, alocacoes_grandes = salva_resultados_penalizacao(resumo_obj_grandes, resumo_grandes, coberturas_grandes, alocacoes_grandes, lambda_pen, results, 'Grandes')

In [ ]:
def plot_fo_lambda(resultado, lambda_vals, MAX_COB, COB_ATUAL, nome):

  # Configurando o estilo
  sns.set_theme(style = "whitegrid")
  fig, ax = plt.subplots(figsize = (12, 7))

  # Linhas de referência
  menor_cob = resultado['FO Cobertura Real'].min()
  maior_cob = resultado['FO Cobertura Real'].max()

  if abs(menor_cob - COB_ATUAL) < 50 or abs(maior_cob - COB_ATUAL) < 50:
    ax.axhline(
        y = COB_ATUAL,
        color = 'red',
        linestyle = '--',
        linewidth = 2,
        label = "Cobertura Atual",
        alpha = 0.5,
        zorder = 0
    )

  if abs(menor_cob - MAX_COB) < 50 or abs(maior_cob - MAX_COB) < 50:
    ax.axhline(
        y = MAX_COB,
        color = 'green',
        linestyle = '--',
        linewidth = 2,
        label = "Máxima Cobertura",
        alpha = 0.5,
        zorder = 0
    )

  # Gráfico de linhas e pontos para as penalizações
  sns.lineplot(
    data = resultado,
    x = "Novas Implantações",
    y = "FO Cobertura Real",
    marker = 'o',
    sort = False,
    color = 'teal',
    linewidth = 2,
    markersize = 8,
    ax = ax
  )

  # Valores da penalização
  dados_filtrados = resultado[resultado['Lambda'].isin(lambda_vals)]

  for _, row in dados_filtrados.iterrows():
    x_pos = row["Novas Implantações"]
    y_pos = row["FO Cobertura Real"]
    lambda_val = row["Lambda"]
    texto = f'λ={lambda_val:.1f}'

    ax.text(
      x = x_pos - 0.05,
      y = y_pos + 0.05,
      s = texto,
      fontsize = 10,
      fontweight = 'bold',
      ha = 'center',
      va = 'bottom'
    )

  ax.set_title("Cobertura Real vs. Realocação de Soros", fontsize = 15)
  ax.set_xlabel("Número de Realocações", fontsize = 12)
  ax.set_ylabel("F.O. Cobertura Real", fontsize = 12)

  plt.tight_layout()

  plt.savefig(f'Figuras/analise-lambda-{nome}.png', dpi = 300, bbox_inches = 'tight')
  plt.show()

  return

In [ ]:
int_cob_atual = int(COB_ATUAL)
cob_sem_penalizacao = resumo_obj_peq[resumo_obj_peq['Lambda'] == 0]['FO Cobertura Real'].item()
MAIOR_COB = int(cob_sem_penalizacao)

escolhidos_pequenos = [0.0, 0.5, 1.0, 1.5, 2.0]
escolhidos_grandes = [2, 4 , 6, 8, 10, 14, 20, 32]

plot_fo_lambda(resumo_obj_peq, escolhidos_pequenos, MAIOR_COB, COB_ATUAL, 'pequenos')

plot_fo_lambda(resumo_obj_grandes, escolhidos_grandes, MAIOR_COB, COB_ATUAL, 'grandes')

In [ ]:
def plot_resultados_lambda(lambda_pen, gdf_cobertura, gdf_alocacao, titulo,
                           tab_cobertura_atual, tab_cobertura_modelo, nome):

  fo_ideal = len(municipios) * len(soros)
  # Corrigido: Usar o nome da coluna com o sufixo lambda_pen
  fo_atual = sum(gdf_cobertura[f'{soro}_{lambda_pen}'].sum() for soro in soros)

  cmap = mcolors.ListedColormap(['white', '#1f4e79'])

  fig, ax = plt.subplots(2, 4, figsize = (35,20))
  ax = ax.flatten()

  for k, soro in enumerate(soros):
    # Contorno dos municípios
    municipios.boundary.plot(ax = ax[k], color = 'dimgray', linewidth = 0.6, zorder = 2)

    coluna_soro = f'{soro}_{lambda_pen}'
    # Colore os municípios cobertos
    gdf_cobertura.plot(ax = ax[k], column = coluna_soro, cmap = cmap, linewidth = 0, vmin = 0, vmax = 1, zorder = 1)

    # Postos com soro
    postos_selecionados = gdf_alocacao[gdf_alocacao[coluna_soro] == 1]

    # Usa o índice de gdf_alocacao (que deve corresponder a J) para mapear para h
    postos_mantidos = postos_selecionados[postos_selecionados.index.map(lambda j_idx: h[j_idx, k] == 1)]
    postos_novos = postos_selecionados[postos_selecionados.index.map(lambda j_idx: h[j_idx, k] == 0)]

    # Plota Postos Mantidos (Amarelo)
    if not postos_mantidos.empty:
        postos_mantidos.plot(
            ax=ax[k], color='#ffd700', markersize=40,
            edgecolor='black', linewidth=0.8, zorder=4
        )

    # Plota Postos Novos (Magenta)
    if not postos_novos.empty:
        postos_novos.plot(
            ax=ax[k], color='#ff00ff', markersize=40,
            edgecolor='black', linewidth=1.0, zorder=5
        )

    # 3. Legenda manual
    legenda_manual = [
        Line2D([0], [0], marker='s', color='w', label='Município não coberto',
        markerfacecolor='white', markersize=12, markeredgecolor='black')
    ]
    if not postos_mantidos.empty:
      legenda_manual.append(
        Line2D([0], [0], marker='o', color='w', label='Soro Mantido',
        markerfacecolor='#ffd700', markersize=10, markeredgecolor='black'),
      )
    if not postos_novos.empty:
      legenda_manual.append(
          Line2D([0], [0], marker='o', color='w', label='Soro Realocado',
        markerfacecolor='#ff00ff', markersize=10, markeredgecolor='black')
      )

    # ax[k].legend(handles=legenda_manual, loc='lower left', fontsize=11, frameon=True)
    ax[k].legend(handles=legenda_manual, loc='lower left', fontsize=16, frameon=True)
    ax[k].axis('off')

    perc_atual = round(tab_cobertura_atual.loc[soro, 'Cobertura (%)'],2)
    perc_modelo = round(tab_cobertura_modelo.loc[soro, 'Cobertura (%)'],2)
    titulo_k = (
    rf"$\bf{{Soro\ {soro}}}$" + "\n"
    rf"{perc_atual:.2f}% $\rightarrow$ {perc_modelo:.2f}%"
)
    # titulo_k = f'Soro {soro}\nRede Atual = {perc_atual}% | Modelo = {perc_modelo}%'

    # ax[k].set_title(titulo_k, fontsize=18, fontweight='bold')
    ax[k].set_title(titulo_k, fontsize=30)

  # fig.suptitle(f'{titulo}\nFO Cobertura: {fo_atual} | Eficiência: {(fo_atual/fo_ideal)*100:.2f}%',
  #                fontsize = 26, fontweight = "bold", y = 0.95)
  fig.suptitle(f'{titulo}\nFO Cobertura: {fo_atual} | Eficiência: {(fo_atual/fo_ideal)*100:.2f}%',
                 fontsize = 36, fontweight = "bold", y = 0.95)

  plt.tight_layout()
  plt.savefig(f"Figuras/modelo-{nome}.png", dpi = 300, bbox_inches='tight')
  plt.show()

In [ ]:
FO_BEST = resumo_obj_peq['FO Cobertura Real']

lambda_best = 0.2
cols_soro_best = [f'{soro}_{lambda_best}' for soro in soros]

cols_cob = ['Municipio', 'Codigo Municipio'] + cols_soro_best + ['geometry']
cobertura_best = coberturas_pequenos[cols_cob]

cols_aloc = ['Nome Posto'] + cols_soro_best + ['geometry']
alocacao_best = alocacoes_pequenos[cols_aloc]

plot_resultados_lambda(
    lambda_best,
    cobertura_best,
    alocacao_best,
    f'Distribuição Otimizada com Demanda ($\\lambda$ = {lambda_best})', # Corrigido: Usando lambda_best
    tab_cobertura_atual,
    resumo_pequenos[resumo_pequenos['Lambda'] == 0.2].set_index('Soro'),
    nome = 'penalizacao'
)